# Whose eyes should a VR stream learn from?

This notebook runs the project from a fresh machine: download the head-tracking data, parse it, and compare ways of picking earlier viewers.

To finish in reasonable time on Colab's two cores it uses fewer trials per video than the README (20 against 200) and writes to `results_quick/`. So the numbers here will be close to the README's but will not match them digit for digit. The README's own numbers are the files under `results/`.

Expect about 45 minutes in total. Parsing the 864 sessions is the slow part.

## 1. Get the code and install what it needs

In [ ]:
import os, subprocess, sys
REPO = os.environ.get('VR_WHOSE_EYES_REPO', 'https://github.com/FireCloud24/vr-whose-eyes.git')
if not os.path.exists('scripts/download.py'):
    if not os.path.exists('vr-whose-eyes'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO, 'vr-whose-eyes'], check=True)
    os.chdir('vr-whose-eyes')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(os.getcwd())

In [ ]:
def run(module, *args):
    """Run one of the project's scripts and stop the notebook if it fails."""
    out = subprocess.run([sys.executable, '-m', module, *map(str, args)], capture_output=True, text=True)
    print(out.stdout[-3000:])
    if out.returncode != 0:
        print(out.stderr[-3000:])
        raise RuntimeError(f'{module} failed with status {out.returncode}')

from IPython.display import Image, Markdown, display
from pathlib import Path
OUT = 'results_quick'
show = lambda name: display(Markdown(Path(OUT, name).read_text(encoding='utf-8')))

## 2. Download the data and check viewer numbers

The zip is about 181 MB. The check looks at when each session was recorded and at headset height, to see whether a viewer folder is one person throughout.

In [ ]:
run('scripts.download')
run('scripts.check_ids', '--out', OUT)
show('id_linkage.md')

## 3. Parse every session and draw the attention heatmaps

This is the slow cell.

In [ ]:
run('scripts.heatmaps', '--out', OUT)
show('data_summary.md')
display(Image(filename=f'{OUT}/heatmaps/all_videos.png'))

## 4. Baselines: own motion, random crowds, and a blend

In [ ]:
run('scripts.baselines', '--out', OUT, '--trials', 20, '--tuning-trials', 5, '--jobs', 2)
show('budget.md')
show('baselines.md')
display(Image(filename=f'{OUT}/crowd_curve.png'))

## 5. Viewing styles, then chosen groups against random groups

The script checks that style is a stable trait before it computes any outcome.

In [ ]:
run('scripts.chosen_vs_random', '--out', OUT, '--trials', 20, '--jobs', 2)
show('style_check.md')
show('chosen_vs_random.md')
display(Image(filename=f'{OUT}/chosen_vs_random.png'))

## 6. Four chosen viewers against sixteen random ones

In [ ]:
run('scripts.four_vs_sixteen', '--out', OUT)
show('four_vs_sixteen.md')
display(Image(filename=f'{OUT}/four_vs_sixteen.png'))

## 7. A few of the tests

The geometry, metric and predictor tests need no data and run in seconds. `python -m pytest -q` runs everything.

In [ ]:
out = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_geometry.py', 'tests/test_metrics.py', 'tests/test_predict.py', 'tests/test_evaluate.py'], capture_output=True, text=True)
print(out.stdout[-1500:])
assert out.returncode == 0